<a href="https://colab.research.google.com/github/AtharvaSiddhawar/KIRA-AI-Assistant/blob/main/notebooks/easy_training_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Training a microWakeWord Model

<div style="background-color: #f0f7fb; padding: 15px; border-radius: 10px; border-left: 5px solid #3498db; margin-bottom: 20px;">
    <h2 style="margin-top: 0; color: #3498db;">Welcome to microWakeWord Training!</h2>
    <p>This notebook steps you through training a basic microWakeWord model. It is intended as a <b>starting point</b> for users who want to create their own wake word model. You should use <b>Python 3.10</b> for the best experience.</p>
    <p>The training process follows these main steps:</p>
    <ol>
        <li>Setup the environment and install dependencies</li>
        <li>Generate wake word samples using text-to-speech</li>
        <li>Download and prepare background audio for training</li>
        <li>Set up audio augmentation to create robust training data</li>
        <li>Configure and train the neural network model</li>
        <li>Export the model for use with ESPHome</li>
    </ol>
</div>

<div style="background-color: #fff3cd; padding: 15px; border-radius: 10px; border-left: 5px solid #f0ad4e; margin-bottom: 20px;">
    <h3 style="margin-top: 0; color: #8a6d3b;">⚠️ Important Note</h3>
    <p>The model generated will most likely not be usable for everyday use without experimentation; it may be difficult to trigger or falsely activate too frequently. You will most likely have to experiment with many different settings to obtain a decent model!</p>
</div>

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

## 📦 Step 1: Setup Environment

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
    <p><b>What this step does:</b> Installs all necessary dependencies for microWakeWord training, including platform-specific requirements.</p>
    <p><b>Expected time:</b> 2-5 minutes depending on your internet connection</p>
    <p><b>Note:</b> You may need to restart your notebook kernel after this step completes.</p>
</div>

In [3]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
import platform

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

# Install ipywidgets for interactive notebook elements
!pip install ipywidgets

!git clone https://github.com/BigPappy098/microWakeWord
!pip install --ignore-requires-python -e ./microWakeWord

  Cloning https://github.com/whatsnowplaying/audio-metadata (to revision d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f) to /tmp/pip-req-build-dvc91h0r
  Running command git clone --filter=blob:none --quiet https://github.com/whatsnowplaying/audio-metadata /tmp/pip-req-build-dvc91h0r
  Running command git rev-parse -q --verify 'sha^d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
  Running command git fetch -q https://github.com/whatsnowplaying/audio-metadata d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Running command git checkout -q d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Resolved https://github.com/whatsnowplaying/audio-metadata to commit d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
fatal: destination path 'microWakeWord' already exists and is not an empty directory.
Obtaining file:///content/microWakeWord
  Installing build dependencies ... done
  Checking if build

ModuleNotFoundError: No module named 'microwakeword'

In [2]:
import sys
import microwakeword

print("Python:", sys.version)
print("microWakeWord import: OK")
print("Package:", microwakeword.__file__)

ModuleNotFoundError: No module named 'microwakeword'

In [3]:
# Clean clone of the current repo
!rm -rf /content/microWakeWord
!git clone https://github.com/FutureProofHomes/microWakeWord.git /content/microWakeWord

# Install into THIS notebook kernel
%pip install --ignore-requires-python /content/microWakeWord

# Make the source package directly visible too
import sys
sys.path.insert(0, "/content/microWakeWord")

# Test
import microwakeword

print("Python:", sys.version)
print("microWakeWord import: OK")
print("Package:", microwakeword.__file__)

Cloning into '/content/microWakeWord'...
remote: Enumerating objects: 396, done.
remote: Counting objects: 100% (396/396), done.
remote: Compressing objects: 100% (232/232), done.
remote: Total 396 (delta 221), reused 318 (delta 158), pack-reused 0 (from 0)
Receiving objects: 100% (396/396), 675.34 KiB | 18.25 MiB/s, done.
Resolving deltas: 100% (221/221), done.
Processing ./microWakeWord
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.5/84.5 kB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

## 🔊 Step 2: Generate Wake Word Samples

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
    <p><b>What this step does:</b> Generates a single sample of your wake word using text-to-speech so you can verify it sounds correct.</p>
    <p><b>Key parameter to modify:</b></p>
    <ul>
        <li><code>target_word</code> - Set this to your desired wake word (use underscores instead of spaces)</li>
    </ul>
    <p><b>Tips:</b></p>
    <ul>
        <li>Try phonetic spellings for better pronunciation (e.g., "hey_komputer" instead of "hey_computer")</li>
        <li>Listen to the generated sample to verify it sounds correct before proceeding</li>
    </ul>
</div>

In [35]:
# Generates 1 sample of the target word for manual verification.

target_word = 'hey_cutie'  # Phonetic spellings may produce better samples

import os
import sys
import platform

from IPython.display import Audio

if not os.path.exists("./piper-sample-generator"):
    if platform.system() == "Darwin":
        !git clone -b mps-support https://github.com/kahrendt/piper-sample-generator
    else:
        !git clone https://github.com/rhasspy/piper-sample-generator

    !wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

    # Install system dependencies
    !pip install torch torchaudio piper-phonemize-cross==1.2.1

    if "piper-sample-generator/" not in sys.path:
        sys.path.append("piper-sample-generator/")

!python3 piper-sample-generator/generate_samples.py "{target_word}" \
--max-samples 1 \
--batch-size 1 \
--output-dir generated_samples

Audio("generated_samples/0.wav", autoplay=True)

python3: can't open file '/content/piper-sample-generator/generate_samples.py': [Errno 2] No such file or directory


ValueError: rate must be specified when data is a numpy array or list of audio samples.

In [1]:
# KIRA - Hey Elli sample generator
from IPython.display import Audio
import os

target_phrase = "hey cutie"

# Install the CURRENT Piper sample generator
%pip install -U piper-sample-generator

# Download the multi-speaker LibriTTS generator model
!mkdir -p /content/piper_models
!wget -q -O /content/piper_models/en_US-libritts_r-medium.pt \
"https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"

# Clear any broken/old sample
!rm -rf /content/generated_samples

# Generate ONE test sample
!python3 -m piper_sample_generator "{target_phrase}" \
  --model /content/piper_models/en_US-libritts_r-medium.pt \
  --max-samples 1 \
  --batch-size 1 \
  --output-dir /content/generated_samples

print("Generated sample:")
Audio("/content/generated_samples/0.wav", autoplay=True)

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/piper_sample_generator/__main__.py", line 20, in <module>
    from piper_train.vits import commons
ModuleNotFoundError: No module named 'piper_train'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.13/dist-packages/piper_sample_generator/__main__.py", line 22, in <module>
    from piper_train.vits import commons
ModuleNotFoundError: No module named 'piper_train'
Generated sample:


ValueError: rate must be specified when data is a numpy array or list of audio samples.

In [2]:
# KIRA - Hey Elli sample generator FIXED
from IPython.display import Audio
import os

target_phrase = "hey cutie"

# Remove any previous source clone
!rm -rf /content/piper-sample-generator

# Clone the FULL source repo.
# This is important because it includes the missing piper_train folder.
!git clone https://github.com/rhasspy/piper-sample-generator.git /content/piper-sample-generator

# Make sure dependencies are installed
%pip install piper-tts==1.3.0

# Download the LibriTTS multi-speaker generator into its expected model folder
!wget -q -O /content/piper-sample-generator/models/en_US-libritts_r-medium.pt \
"https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"

# Remove old failed output
!rm -rf /content/generated_samples

# Run from the SOURCE repo so piper_train is visible
!cd /content/piper-sample-generator && \
PYTHONPATH=/content/piper-sample-generator \
python3 -m piper_sample_generator "{target_phrase}" \
--model /content/piper-sample-generator/models/en_US-libritts_r-medium.pt \
--max-samples 1 \
--batch-size 1 \
--output-dir /content/generated_samples

# Confirm the WAV really exists before trying to play it
assert os.path.exists("/content/generated_samples/0.wav"), \
    "Sample was not generated — check the error above."

print("✅ Hey Elli test sample generated!")
Audio("/content/generated_samples/0.wav", autoplay=True)

Cloning into '/content/piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 184 (delta 70), reused 53 (delta 53), pack-reused 98 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 20.91 MiB/s, done.
Resolving deltas: 100% (93/93), done.
DEBUG:__main__:Loading /content/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/1 complete
INFO:__main__:Done
✅ Hey Elli test sample generated!


In [4]:
# KIRA - HEY CUTIE PRONUNCIATION AUDITION

import os
import glob
import shutil
import subprocess
from IPython.display import Audio, display

# Find the Piper LibriTTS model
models = glob.glob(
    "/content/**/en_US-libritts_r-medium.pt",
    recursive=True
)

assert models, "Piper LibriTTS model not found"

MODEL = models[0]
OUT = "/content/hey_cutie_pronunciation_test"

print("Using model:", MODEL)

if os.path.exists(OUT):
    shutil.rmtree(OUT)

os.makedirs(OUT)

# We are deliberately testing alternate spellings
# because the wake model learns the SOUND, not the written spelling.
tests = {
    "A": "Hey cutie.",
    "B": "Hey, cutie.",
    "C": "Hey cutey.",
    "D": "Hey, cutey.",
    "E": "Hey kyootie.",
    "F": "Hey kyoo tee.",
}

for name, phrase in tests.items():

    folder = os.path.join(OUT, name)

    cmd = [
        "python",
        "-m",
        "piper_sample_generator",
        phrase,
        "--model",
        MODEL,
        "--max-samples",
        "1",
        "--max-speakers",
        "1",
        "--length-scales",
        "1.0",
        "--output-dir",
        folder,
    ]

    print("Generating", name, "->", phrase)

    subprocess.run(
        cmd,
        check=True
    )

print("\n✅ All pronunciation tests generated\n")

for name, phrase in tests.items():

    wav = os.path.join(
        OUT,
        name,
        "0.wav"
    )

    print("============================")
    print(name, ":", phrase)

    display(
        Audio(
            wav,
            autoplay=False
        )
    )

Using model: /content/piper-sample-generator/models/en_US-libritts_r-medium.pt
Generating A -> Hey cutie.


CalledProcessError: Command '['python', '-m', 'piper_sample_generator', 'Hey cutie.', '--model', '/content/piper-sample-generator/models/en_US-libritts_r-medium.pt', '--max-samples', '1', '--max-speakers', '1', '--length-scales', '1.0', '--output-dir', '/content/hey_cutie_pronunciation_test/A']' returned non-zero exit status 1.

In [8]:
# ============================================================
# KIRA - HEY CUTIE PRONUNCIATION TEST
# ============================================================

import os
import shutil
import subprocess
from IPython.display import Audio, display

OUT = "/content/hey_cutie_pronunciation_test"

if os.path.exists(OUT):
    shutil.rmtree(OUT)

os.makedirs(OUT)

tests = {
    "A": "Hey cutie.",
    "B": "Hey, cutie.",
    "C": "Hey cutey.",
    "D": "Hey, cutey.",
    "E": "Hey kyootie.",
    "F": "Hey kyoo tee.",
}

for name, phrase in tests.items():

    folder = os.path.join(
        OUT,
        name
    )

    subprocess.run(
        [
            "python",
            "-m",
            "piper_sample_generator",
            phrase,
            "--model",
            MODEL,
            "--max-samples",
            "1",
            "--max-speakers",
            "1",
            "--output-dir",
            folder,
        ],
        check=True
    )

print("\n✅ Samples ready\n")

for name, phrase in tests.items():

    wav = os.path.join(
        OUT,
        name,
        "0.wav"
    )

    print(name, "→", phrase)

    display(
        Audio(
            wav,
            autoplay=False
        )
    )

CalledProcessError: Command '['python', '-m', 'piper_sample_generator', 'Hey cutie.', '--model', '/content/piper-sample-generator/models/en_US-libritts_r-medium.pt', '--max-samples', '1', '--max-speakers', '1', '--output-dir', '/content/hey_cutie_pronunciation_test/A']' returned non-zero exit status 1.

In [9]:
# ============================================================
# KIRA - HEY CUTIE PRONUNCIATION TEST
# ============================================================

import os
import shutil
import subprocess
from IPython.display import Audio, display

OUT = "/content/hey_cutie_pronunciation_test"

if os.path.exists(OUT):
    shutil.rmtree(OUT)

os.makedirs(OUT)

tests = {
    "A": "Hey cutie.",
    "B": "Hey, cutie.",
    "C": "Hey cutey.",
    "D": "Hey, cutey.",
    "E": "Hey kyootie.",
    "F": "Hey kyoo tee.",
}

for name, phrase in tests.items():

    folder = os.path.join(
        OUT,
        name
    )

    subprocess.run(
        [
            "python",
            "-m",
            "piper_sample_generator",
            phrase,
            "--model",
            MODEL,
            "--max-samples",
            "1",
            "--max-speakers",
            "1",
            "--output-dir",
            folder,
        ],
        check=True
    )

print("\n✅ Samples ready\n")

for name, phrase in tests.items():

    wav = os.path.join(
        OUT,
        name,
        "0.wav"
    )

    print(name, "→", phrase)

    display(
        Audio(
            wav,
            autoplay=False
        )
    )

CalledProcessError: Command '['python', '-m', 'piper_sample_generator', 'Hey cutie.', '--model', '/content/piper-sample-generator/models/en_US-libritts_r-medium.pt', '--max-samples', '1', '--max-speakers', '1', '--output-dir', '/content/hey_cutie_pronunciation_test/A']' returned non-zero exit status 1.

In [10]:
# FIX THE MISSING LIBRITTS CONFIG

import os
import urllib.request

MODEL = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt"
CONFIG = MODEL + ".json"

os.makedirs(os.path.dirname(MODEL), exist_ok=True)

config_url = (
    "https://raw.githubusercontent.com/"
    "rhasspy/piper-sample-generator/master/"
    "models/en_US-libritts_r-medium.pt.json"
)

urllib.request.urlretrieve(
    config_url,
    CONFIG
)

print("✅ Model:", os.path.exists(MODEL))
print("✅ Config:", os.path.exists(CONFIG))
print("Config size:", os.path.getsize(CONFIG), "bytes")

✅ Model: True
✅ Config: True
Config size: 25782 bytes


In [11]:
# ============================================================
# KIRA - HEY CUTIE PRONUNCIATION TEST
# ============================================================

import os
import shutil
import subprocess
from IPython.display import Audio, display

OUT = "/content/hey_cutie_pronunciation_test"

if os.path.exists(OUT):
    shutil.rmtree(OUT)

os.makedirs(OUT)

tests = {
    "A": "Hey cutie.",
    "B": "Hey, cutie.",
    "C": "Hey cutey.",
    "D": "Hey, cutey.",
    "E": "Hey kyootie.",
    "F": "Hey kyoo tee.",
}

for name, phrase in tests.items():

    folder = os.path.join(
        OUT,
        name
    )

    subprocess.run(
        [
            "python",
            "-m",
            "piper_sample_generator",
            phrase,
            "--model",
            MODEL,
            "--max-samples",
            "1",
            "--max-speakers",
            "1",
            "--output-dir",
            folder,
        ],
        check=True
    )

print("\n✅ Samples ready\n")

for name, phrase in tests.items():

    wav = os.path.join(
        OUT,
        name,
        "0.wav"
    )

    print(name, "→", phrase)

    display(
        Audio(
            wav,
            autoplay=False
        )
    )

CalledProcessError: Command '['python', '-m', 'piper_sample_generator', 'Hey cutie.', '--model', '/content/piper-sample-generator/models/en_US-libritts_r-medium.pt', '--max-samples', '1', '--max-speakers', '1', '--output-dir', '/content/hey_cutie_pronunciation_test/A']' returned non-zero exit status 1.

In [12]:
# ============================================================
# KIRA - HEY CUTIE PRONUNCIATION TEST - FIXED
# ============================================================

import os
import sys
import shutil
import subprocess
from IPython.display import Audio, display

ROOT = "/content/piper-sample-generator"

MODEL = (
    ROOT +
    "/models/en_US-libritts_r-medium.pt"
)

CONFIG = MODEL + ".json"

# ------------------------------------------------------------
# Verify required files
# ------------------------------------------------------------

assert os.path.exists(ROOT), \
    "❌ piper-sample-generator folder missing"

assert os.path.exists(MODEL), \
    "❌ Piper model missing"

assert os.path.exists(CONFIG), \
    "❌ Piper model config missing"

assert os.path.isdir(
    ROOT + "/piper_train"
), "❌ piper_train source folder missing"


print("✅ Piper repository found")
print("✅ Model found")
print("✅ Config found")
print("✅ piper_train found")


# ------------------------------------------------------------
# Make cloned repository visible to Python
# ------------------------------------------------------------

env = os.environ.copy()

env["PYTHONPATH"] = (
    ROOT +
    os.pathsep +
    env.get("PYTHONPATH", "")
)


# ------------------------------------------------------------
# Pronunciation candidates
# ------------------------------------------------------------

tests = {
    "A": "Hey cutie.",
    "B": "Hey, cutie.",
    "C": "Hey cutey.",
    "D": "Hey, cutey.",
    "E": "Hey kyootie.",
    "F": "Hey kyoo tee.",
}

OUT = "/content/hey_cutie_pronunciation_test"

if os.path.exists(OUT):
    shutil.rmtree(OUT)

os.makedirs(OUT)


# ------------------------------------------------------------
# Generate
# ------------------------------------------------------------

for name, phrase in tests.items():

    folder = os.path.join(
        OUT,
        name
    )

    print(
        f"\nGenerating {name}: {phrase}"
    )

    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "piper_sample_generator",

            phrase,

            "--model",
            MODEL,

            "--max-samples",
            "1",

            "--max-speakers",
            "1",

            "--length-scales",
            "1.0",

            "--output-dir",
            folder,
        ],

        cwd=ROOT,
        env=env,

        text=True,
        capture_output=True
    )

    if result.returncode != 0:

        print("\n❌ PIPER FAILED")
        print("==============================")
        print(result.stdout)
        print(result.stderr)
        print("==============================")

        raise RuntimeError(
            f"Piper failed on sample {name}"
        )

    print("✅ Generated")


# ------------------------------------------------------------
# Play all candidates
# ------------------------------------------------------------

print("\n==============================")
print("✅ ALL PRONUNCIATIONS READY")
print("==============================\n")

for name, phrase in tests.items():

    wav = os.path.join(
        OUT,
        name,
        "0.wav"
    )

    assert os.path.exists(wav), \
        f"Missing {wav}"

    print(name, "→", phrase)

    display(
        Audio(
            wav,
            autoplay=False
        )
    )

AssertionError: ❌ piper_train source folder missing

In [6]:
# FIX THE MISSING LIBRITTS CONFIG

import os
import urllib.request

MODEL = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt"
CONFIG = MODEL + ".json"

os.makedirs(os.path.dirname(MODEL), exist_ok=True)

config_url = (
    "https://raw.githubusercontent.com/"
    "rhasspy/piper-sample-generator/master/"
    "models/en_US-libritts_r-medium.pt.json"
)

urllib.request.urlretrieve(
    config_url,
    CONFIG
)

print("✅ Model:", os.path.exists(MODEL))
print("✅ Config:", os.path.exists(CONFIG))
print("Config size:", os.path.getsize(CONFIG), "bytes")

✅ Model: True
✅ Config: True
Config size: 25782 bytes


In [2]:
# ============================================================
# KIRA - RESTORE PIPER LIBRITTS MODEL
# ============================================================

import os
import glob
import subprocess

MODEL_DIR = "/content/piper-sample-generator/models"
MODEL = os.path.join(
    MODEL_DIR,
    "en_US-libritts_r-medium.pt"
)

os.makedirs(MODEL_DIR, exist_ok=True)

# Search entire runtime first
found = glob.glob(
    "/content/**/en_US-libritts_r-medium.pt",
    recursive=True
)

if found:
    MODEL = found[0]
    print("✅ Existing Piper model found:")
    print(MODEL)

else:
    print("⬇️ Piper model missing - downloading again...")

    url = (
        "https://github.com/rhasspy/"
        "piper-sample-generator/releases/download/"
        "v2.0.0/en_US-libritts_r-medium.pt"
    )

    subprocess.run(
        [
            "wget",
            "-O",
            MODEL,
            url
        ],
        check=True
    )

    print("✅ Piper model downloaded")

print()
print("Model:", MODEL)
print(
    "Size:",
    round(os.path.getsize(MODEL) / 1024 / 1024, 1),
    "MB"
)

⬇️ Piper model missing - downloading again...
✅ Piper model downloaded

Model: /content/piper-sample-generator/models/en_US-libritts_r-medium.pt
Size: 194.6 MB


In [7]:
# KIRA - Generate 1000 "Hey Elli" training samples

import os
from pathlib import Path

target_phrase = "hey elli"
output_dir = "/content/generated_samples"

# Remove the single test sample so we start clean
!rm -rf "{output_dir}"

# Generate 1000 varied samples
!cd /content/piper-sample-generator && \
PYTHONPATH=/content/piper-sample-generator \
python3 -m piper_sample_generator "{target_phrase}" \
--model /content/piper-sample-generator/models/en_US-libritts_r-medium.pt \
--max-samples 1000 \
--batch-size 100 \
--output-dir "{output_dir}"

# Verify how many WAV files were created
files = list(Path(output_dir).glob("*.wav"))

print()
print("✅ Generated samples:", len(files))

assert len(files) == 1000, \
    f"Expected 1000 samples, but found {len(files)}"

DEBUG:__main__:Loading /content/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/10 complete
DEBUG:__main__:Batch 2/10 complete
DEBUG:__main__:Batch 3/10 complete
DEBUG:__main__:Batch 4/10 complete
DEBUG:__main__:Batch 5/10 complete
DEBUG:__main__:Batch 6/10 complete
DEBUG:__main__:Batch 7/10 complete
DEBUG:__main__:Batch 8/10 complete
DEBUG:__main__:Batch 9/10 complete
DEBUG:__main__:Batch 10/10 complete
INFO:__main__:Done

✅ Generated samples: 1000


In [8]:
# Backup our 1000 Hey Elli samples
!cd /content && zip -q -r hey_elli_1000_samples.zip generated_samples

import os
size_mb = os.path.getsize("/content/hey_elli_1000_samples.zip") / (1024 * 1024)

print(f"✅ Backup created: {size_mb:.1f} MB")

✅ Backup created: 23.5 MB


### 🔊 Step 2.1: Generate Multiple Wake Word Samples

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
    <p><b>What this step does:</b> Generates a larger set of wake word samples (1000 by default) for training.</p>
    <p><b>Key parameters to modify:</b></p>
    <ul>
        <li><code>--max-samples</code> - Number of samples to generate (default: 1000)</li>
        <li><code>--batch-size</code> - How many samples to generate at once (default: 100)</li>
    </ul>
    <p><b>Advanced options:</b> See the <a href="https://github.com/rhasspy/piper-sample-generator">piper-sample-generator documentation</a> for additional parameters like:</p>
    <ul>
        <li><code>--noise-scale</code> - Controls voice variation (higher = more variation)</li>
        <li><code>--noise-w</code> - Controls speaking style variation</li>
        <li><code>--length-scale</code> - Controls speaking speed (higher = slower)</li>
    </ul>
</div>

In [ ]:
# Generates a larger amount of wake word samples.
# Start here when trying to improve your model.
# See https://github.com/rhasspy/piper-sample-generator for the full set of
# parameters. In particular, experiment with noise-scales and noise-scale-ws,
# generating negative samples similar to the wake word, and generating many more
# wake word samples, possibly with different phonetic pronunciations.

!python3 piper-sample-generator/generate_samples.py "{target_word}" \
--max-samples 1000 \
--batch-size 100 \
--output-dir generated_samples

## 🎵 Step 3: Download Background Audio Data

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
    <p><b>What this step does:</b> Downloads audio data for augmentation, including room impulse responses and background noise.</p>
    <p><b>Expected time:</b> 10-20 minutes (this step can be slow!)</p>
    <p><b>Why this matters:</b> Good background audio is essential for training a robust wake word model that works in real environments.</p>
    <p><b>Note:</b> The data downloaded has mixed licenses and should be considered for <b>non-commercial personal use only</b>.</p>
</div>

In [9]:
# Downloads audio data for augmentation. This can be slow!
# Borrowed from openWakeWord's automatic_model_training.ipynb, accessed March 4, 2024
#
# **Important note!** The data downloaded here has a mixture of difference
# licenses and usage restrictions. As such, any custom models trained with this
# data should be considered as appropriate for **non-commercial** personal use only.


import datasets
import scipy
import os

import numpy as np

from pathlib import Path
from tqdm import tqdm

## Download MIR RIR data

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    # Save clips to 16-bit PCM wav files
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"
    link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
    !wget -O {out_dir} {link}
    !cd audioset && tar -xf bal_train09.tar

    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset
# https://github.com/mdeff/fma
# (Third-party mchl914 extra small set)

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fname = "fma_xs.zip"
    link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
    out_dir = f"fma/{fname}"
    !wget -O {out_dir} {link}
    !cd {output_dir} && unzip -q {fname}

    output_dir = "./fma_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    fma_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]})
    fma_dataset = fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(fma_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))


README.md:   0%|          | 0.00/936 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

0it [00:05, ?it/s]


TypeError: 'torchcodec.decoders.AudioDecoder' object is not subscriptable

In [10]:
# KIRA - Step 3 compatibility fix for current Hugging Face Datasets
# Downloads + prepares RIR, AudioSet and FMA background audio at 16 kHz.

import os
import shutil
import tarfile
import zipfile
import subprocess
from pathlib import Path

import datasets
import numpy as np
import scipy.io.wavfile
import scipy.signal

from tqdm.auto import tqdm


TARGET_SR = 16000


def decoder_to_audio(decoder):
    """Convert current HuggingFace AudioDecoder -> mono numpy float32."""
    samples = decoder.get_all_samples()

    audio = samples.data.detach().cpu().numpy()
    sr = int(samples.sample_rate)

    # TorchCodec shape is normally [channels, samples]
    if audio.ndim == 2:
        if audio.shape[0] > 1:
            audio = audio.mean(axis=0)
        else:
            audio = audio[0]

    audio = audio.astype(np.float32)

    # Safety clamp
    audio = np.clip(audio, -1.0, 1.0)

    # Resample only if necessary
    if sr != TARGET_SR:
        gcd = np.gcd(sr, TARGET_SR)

        audio = scipy.signal.resample_poly(
            audio,
            TARGET_SR // gcd,
            sr // gcd
        )

        sr = TARGET_SR

    return audio, sr


def save_decoder_as_wav(decoder, output_path):
    audio, sr = decoder_to_audio(decoder)

    pcm16 = (
        np.clip(audio, -1.0, 1.0) * 32767
    ).astype(np.int16)

    scipy.io.wavfile.write(
        str(output_path),
        sr,
        pcm16
    )


# ============================================================
# 1. MIT ROOM IMPULSE RESPONSES
# ============================================================

print("\n=== MIT room impulse responses ===")

rir_dir = Path("/content/mit_rirs")

# Previous failed attempt may have left a partial directory.
if rir_dir.exists():
    shutil.rmtree(rir_dir)

rir_dir.mkdir(parents=True, exist_ok=True)

rir_dataset = datasets.load_dataset(
    "davidscripka/MIT_environmental_impulse_responses",
    split="train",
    streaming=True
)

rir_dataset = rir_dataset.cast_column(
    "audio",
    datasets.Audio(
        sampling_rate=TARGET_SR,
        num_channels=1
    )
)

rir_count = 0

for i, row in enumerate(tqdm(rir_dataset)):
    save_decoder_as_wav(
        row["audio"],
        rir_dir / f"rir_{i:05d}.wav"
    )

    rir_count += 1

print("✅ RIR files:", rir_count)


# ============================================================
# 2. AUDIOSET BACKGROUND NOISE
# ============================================================

print("\n=== AudioSet background audio ===")

audioset_root = Path("/content/audioset")
audioset_audio = audioset_root / "audio"
audioset_tar = audioset_root / "bal_train09.tar"

audioset_root.mkdir(parents=True, exist_ok=True)

if not audioset_audio.exists():

    if not audioset_tar.exists():
        subprocess.run(
            [
                "wget",
                "-O",
                str(audioset_tar),
                "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar"
            ],
            check=True
        )

    print("Extracting AudioSet...")

    with tarfile.open(audioset_tar) as archive:
        archive.extractall(audioset_root)


audioset_paths = [
    str(p)
    for p in audioset_audio.glob("**/*.flac")
]

print("AudioSet source files:", len(audioset_paths))

audioset_out = Path("/content/audioset_16k")
audioset_out.mkdir(parents=True, exist_ok=True)

audioset_dataset = datasets.Dataset.from_dict(
    {
        "path": audioset_paths,
        "audio": audioset_paths
    }
)

audioset_dataset = audioset_dataset.cast_column(
    "audio",
    datasets.Audio(
        sampling_rate=TARGET_SR,
        num_channels=1
    )
)

for row in tqdm(audioset_dataset):

    source_path = Path(row["path"])

    save_decoder_as_wav(
        row["audio"],
        audioset_out /
        (source_path.stem + ".wav")
    )

print(
    "✅ AudioSet WAVs:",
    len(list(audioset_out.glob("*.wav")))
)


# ============================================================
# 3. FREE MUSIC ARCHIVE
# ============================================================

print("\n=== Free Music Archive ===")

fma_root = Path("/content/fma")
fma_zip = fma_root / "fma_xs.zip"

fma_root.mkdir(parents=True, exist_ok=True)

fma_source = fma_root / "fma_small"

if not fma_source.exists():

    if not fma_zip.exists():
        subprocess.run(
            [
                "wget",
                "-O",
                str(fma_zip),
                "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip"
            ],
            check=True
        )

    print("Extracting FMA...")

    with zipfile.ZipFile(fma_zip, "r") as archive:
        archive.extractall(fma_root)


fma_paths = [
    str(p)
    for p in fma_source.glob("**/*.mp3")
]

print("FMA source files:", len(fma_paths))

fma_out = Path("/content/fma_16k")
fma_out.mkdir(parents=True, exist_ok=True)

fma_dataset = datasets.Dataset.from_dict(
    {
        "path": fma_paths,
        "audio": fma_paths
    }
)

fma_dataset = fma_dataset.cast_column(
    "audio",
    datasets.Audio(
        sampling_rate=TARGET_SR,
        num_channels=1
    )
)

for row in tqdm(fma_dataset):

    source_path = Path(row["path"])

    save_decoder_as_wav(
        row["audio"],
        fma_out /
        (source_path.stem + ".wav")
    )


print(
    "✅ FMA WAVs:",
    len(list(fma_out.glob("*.wav")))
)


print("\n==============================")
print("✅ STEP 3 COMPLETE")
print("==============================")

print("RIR:", len(list(rir_dir.glob('*.wav'))))
print("AudioSet:", len(list(audioset_out.glob('*.wav'))))
print("FMA:", len(list(fma_out.glob('*.wav'))))


=== MIT room impulse responses ===


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

0it [00:00, ?it/s]

✅ RIR files: 270

=== AudioSet background audio ===


CalledProcessError: Command '['wget', '-O', '/content/audioset/bal_train09.tar', 'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar']' returned non-zero exit status 8.

In [11]:
# ============================================================
# KIRA STEP 3 CONTINUATION
# Keep the 270 RIRs already generated.
# Uses CURRENT AudioSet Parquet/streaming format.
# ============================================================

import os
import shutil
from pathlib import Path

import datasets
import numpy as np
import scipy.io.wavfile
import scipy.signal

from tqdm.auto import tqdm


TARGET_SR = 16000

# For our FIRST Hey Elli model, 2500 diverse AudioSet clips is plenty.
# We can increase this later when tuning false activations.
MAX_AUDIOSET_CLIPS = 2500


def decoder_to_audio(decoder):
    samples = decoder.get_all_samples()

    audio = samples.data.detach().cpu().numpy()
    sr = int(samples.sample_rate)

    # Convert stereo/multichannel -> mono
    if audio.ndim == 2:
        audio = audio.mean(axis=0)

    audio = np.asarray(audio).squeeze().astype(np.float32)

    if sr != TARGET_SR:
        gcd = np.gcd(sr, TARGET_SR)

        audio = scipy.signal.resample_poly(
            audio,
            TARGET_SR // gcd,
            sr // gcd
        )

    audio = np.clip(audio, -1.0, 1.0)

    return audio


def save_audio(decoder, path):
    audio = decoder_to_audio(decoder)

    pcm16 = (
        audio * 32767
    ).astype(np.int16)

    scipy.io.wavfile.write(
        str(path),
        TARGET_SR,
        pcm16
    )


# ------------------------------------------------------------
# Remove obsolete failed TAR attempt
# ------------------------------------------------------------

bad_tar = Path("/content/audioset/bal_train09.tar")

if bad_tar.exists():
    bad_tar.unlink()

print("✅ Existing MIT RIRs:",
      len(list(Path("/content/mit_rirs").glob("*.wav"))))


# ============================================================
# AUDIOSET - CURRENT FORMAT
# ============================================================

print("\n=== Streaming current AudioSet balanced/train ===")

audioset_out = Path("/content/audioset_16k")

if audioset_out.exists():
    shutil.rmtree(audioset_out)

audioset_out.mkdir(parents=True, exist_ok=True)


audioset = datasets.load_dataset(
    "agkphysics/AudioSet",
    "balanced",
    split="train",
    streaming=True
)

# Ask HF to decode/resample at 16 kHz mono
audioset = audioset.cast_column(
    "audio",
    datasets.Audio(
        sampling_rate=TARGET_SR,
        num_channels=1
    )
)


count = 0

for row in tqdm(
    audioset,
    total=MAX_AUDIOSET_CLIPS
):
    try:
        save_audio(
            row["audio"],
            audioset_out / f"background_{count:05d}.wav"
        )

        count += 1

    except Exception as e:
        # Skip an occasional bad/missing clip instead of killing Step 3
        print("Skipping one clip:", type(e).__name__)

    if count >= MAX_AUDIOSET_CLIPS:
        break


print("✅ AudioSet background WAVs:", count)

assert count >= 2000, \
    f"Too few AudioSet clips were created: {count}"


# ============================================================
# FINAL STEP-3 CHECK
# ============================================================

rir_count = len(
    list(
        Path("/content/mit_rirs").glob("*.wav")
    )
)

audio_count = len(
    list(
        Path("/content/audioset_16k").glob("*.wav")
    )
)


print("\n================================")
print("✅ KIRA STEP 3 CORE DATA READY")
print("================================")
print("MIT RIRs :", rir_count)
print("AudioSet :", audio_count)

✅ Existing MIT RIRs: 270

=== Streaming current AudioSet balanced/train ===


README.md:   0%|          | 0.00/5.20k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

  0%|          | 0/2500 [00:00<?, ?it/s]

✅ AudioSet background WAVs: 2500

✅ KIRA STEP 3 CORE DATA READY
MIT RIRs : 270
AudioSet : 2500


In [12]:
# ============================================================
# KIRA "Hey Elli" - Step 4: Audio Augmentation
# ============================================================

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

# Our 1000 positive "Hey Elli" samples
clips = Clips(
    input_directory="generated_samples",
    file_pattern="*.wav",
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

# Robust real-world augmentation
augmenter = Augmentation(
    augmentation_duration_s=3.2,

    augmentation_probabilities={
        "SevenBandParametricEQ": 0.10,
        "TanhDistortion":       0.10,
        "PitchShift":           0.10,
        "BandStopFilter":       0.10,
        "AddColorNoise":        0.10,
        "AddBackgroundNoise":   0.75,
        "Gain":                 1.00,
        "RIR":                  0.50,
    },

    # 270 room impulse responses
    impulse_paths=[
        "mit_rirs"
    ],

    # Our 2500 prepared real-world background clips
    background_paths=[
        "audioset_16k"
    ],

    background_min_snr_db=-5,
    background_max_snr_db=10,

    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

print("✅ Hey Elli augmentation configured")
print("Positive samples: 1000")
print("RIRs: 270")
print("Background clips: 2500")

    pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


AttributeError: module 'audiomentations' has no attribute 'AddColorNoise'

In [13]:
# Fix audiomentations version for microWakeWord Step 4
%pip install -U "audiomentations==0.43.1"

  Using cached audiomentations-0.43.1-py3-none-any.whl.metadata (11 kB)
Using cached audiomentations-0.43.1-py3-none-any.whl (86 kB)
  Attempting uninstall: audiomentations
    Found existing installation: audiomentations 0.33.0
    Uninstalling audiomentations-0.33.0:
      Successfully uninstalled audiomentations-0.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
piper-sample-generator 3.2.0 requires audiomentations==0.33.0, but you have audiomentations 0.43.1 which is incompatible.


In [1]:
import audiomentations as aud

print("audiomentations:", aud.__version__)
print("AddColorNoise:", hasattr(aud, "AddColorNoise"))
print("GainTransition:", hasattr(aud, "GainTransition"))
print("ApplyImpulseResponse:", hasattr(aud, "ApplyImpulseResponse"))

audiomentations: 0.43.1
AddColorNoise: True
GainTransition: True
ApplyImpulseResponse: True


## 🔄 Step 4: Set Up Audio Augmentation

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
    <p><b>What this step does:</b> Configures audio augmentation to create more varied training samples.</p>
    <p><b>Why this matters:</b> Augmentation helps the model learn to recognize your wake word in different environments and conditions.</p>
    <p><b>Key parameters to experiment with:</b></p>
    <ul>
        <li><code>augmentation_probabilities</code> - Chances of applying different audio effects</li>
        <li><code>background_min_snr_db</code> and <code>background_max_snr_db</code> - Signal-to-noise ratio range</li>
    </ul>
</div>

In [2]:
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(
    input_directory="generated_samples",
    file_pattern="*.wav",
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,

    augmentation_probabilities={
        "SevenBandParametricEQ": 0.10,
        "TanhDistortion":       0.10,
        "PitchShift":           0.10,
        "BandStopFilter":       0.10,
        "AddColorNoise":        0.10,
        "AddBackgroundNoise":   0.75,
        "Gain":                 1.00,
        "RIR":                  0.50,
    },

    impulse_paths=["mit_rirs"],
    background_paths=["audioset_16k"],

    background_min_snr_db=-5,
    background_max_snr_db=10,

    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

print("✅ Hey Elli augmentation configured")

ModuleNotFoundError: No module named 'microwakeword.audio'

In [3]:
import os
import sys

SOURCE = "/content/microWakeWord"

assert os.path.isdir(SOURCE), "microWakeWord source folder is missing"
assert os.path.isdir(SOURCE + "/microwakeword/audio"), "audio source folder is missing"

# Put source checkout FIRST in Python search path
if SOURCE in sys.path:
    sys.path.remove(SOURCE)

sys.path.insert(0, SOURCE)

# Clear the old/incomplete imported package
for name in list(sys.modules):
    if name == "microwakeword" or name.startswith("microwakeword."):
        del sys.modules[name]

# Test the exact modules we need
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

print("✅ microWakeWord source path fixed")
print("✅ audio modules imported successfully")
print("Source:", SOURCE)

✅ microWakeWord source path fixed
✅ audio modules imported successfully
Source: /content/microWakeWord


    pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


### 🔄 Step 4.1: Test Audio Augmentation

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
    <p><b>What this step does:</b> Augments a random clip and plays it back so you can verify the augmentation sounds reasonable.</p>
    <p><b>What to listen for:</b> The wake word should still be recognizable despite background noise and effects.</p>
    <p><b>Tip:</b> If the augmentation is too strong (wake word not audible) or too weak (no background noise), adjust the parameters in the previous cell.</p>
</div>

In [4]:
from IPython.display import Audio, display
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)

save_clip(
    augmented_clip,
    "hey_elli_augmented_test.wav"
)

print("✅ Augmented Hey Elli sample generated")

display(
    Audio(
        "hey_elli_augmented_test.wav",
        autoplay=False
    )
)

NameError: name 'clips' is not defined

In [5]:
# ============================================================
# KIRA - Rebuild Step 4 + Test Step 4.1
# ============================================================

import os
import sys

# Force Colab to use the full microWakeWord source checkout
SOURCE = "/content/microWakeWord"

if SOURCE in sys.path:
    sys.path.remove(SOURCE)

sys.path.insert(0, SOURCE)

# Clear any incomplete package already loaded
for name in list(sys.modules):
    if name == "microwakeword" or name.startswith("microwakeword."):
        del sys.modules[name]

# Imports
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.audio_utils import save_clip
from IPython.display import Audio, display


# Make sure our data survived the restart
assert os.path.isdir("/content/generated_samples"), \
    "generated_samples folder missing"

assert os.path.isdir("/content/mit_rirs"), \
    "mit_rirs folder missing"

assert os.path.isdir("/content/audioset_16k"), \
    "audioset_16k folder missing"


# ------------------------------------------------------------
# Recreate positive clips object
# ------------------------------------------------------------

clips = Clips(
    input_directory="/content/generated_samples",
    file_pattern="*.wav",
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)


# ------------------------------------------------------------
# Recreate augmentation pipeline
# ------------------------------------------------------------

augmenter = Augmentation(
    augmentation_duration_s=3.2,

    augmentation_probabilities={
        "SevenBandParametricEQ": 0.10,
        "TanhDistortion":       0.10,
        "PitchShift":           0.10,
        "BandStopFilter":       0.10,
        "AddColorNoise":        0.10,
        "AddBackgroundNoise":   0.75,
        "Gain":                 1.00,
        "RIR":                  0.50,
    },

    impulse_paths=[
        "/content/mit_rirs"
    ],

    background_paths=[
        "/content/audioset_16k"
    ],

    background_min_snr_db=-5,
    background_max_snr_db=10,

    min_jitter_s=0.195,
    max_jitter_s=0.205,
)


print("✅ Step 4 rebuilt")
print("✅ clips object ready")
print("✅ augmenter ready")


# ------------------------------------------------------------
# Step 4.1 test
# ------------------------------------------------------------

random_clip = clips.get_random_clip()

augmented_clip = augmenter.augment_clip(
    random_clip
)

test_file = "/content/hey_elli_augmented_test.wav"

save_clip(
    augmented_clip,
    test_file
)

print("✅ Augmented Hey Elli test generated")

display(
    Audio(
        test_file,
        autoplay=False
    )
)

✅ Step 4 rebuilt
✅ clips object ready
✅ augmenter ready
✅ Augmented Hey Elli test generated


## 🔄 Step 5: Generate Augmented Features

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
    <p><b>What this step does:</b> Augments samples and saves training, validation, and testing sets.</p>
    <p><b>Why this matters:</b> This creates the actual data that will be used to train the neural network.</p>
    <p><b>Note:</b> The training set uses more repetition to help the model learn, while the testing set uses a streaming approach to better simulate real-world usage.</p>
</div>

In [8]:
# ============================================================
# KIRA - HEY ELLI
# STEP 5: GENERATE AUGMENTED TRAINING FEATURES
# ============================================================

import os
import sys
import shutil
import random
import numpy as np

SOURCE = "/content/microWakeWord"

# ------------------------------------------------------------
# Force full source checkout
# ------------------------------------------------------------

if SOURCE in sys.path:
    sys.path.remove(SOURCE)

sys.path.insert(0, SOURCE)

for name in list(sys.modules):
    if name == "microwakeword" or name.startswith("microwakeword."):
        del sys.modules[name]


from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap


# ------------------------------------------------------------
# Hugging Face AudioDecoder compatibility
# ------------------------------------------------------------

def audio_entry_to_numpy(entry):

    # Older datasets format
    if isinstance(entry, dict):
        return np.asarray(
            entry["array"],
            dtype=np.float32
        )

    # Current datasets / TorchCodec format
    if hasattr(entry, "get_all_samples"):

        samples = entry.get_all_samples()

        audio = (
            samples.data
            .detach()
            .cpu()
            .numpy()
        )

        # channels -> mono
        if audio.ndim == 2:
            audio = audio.mean(axis=0)

        return np.asarray(
            audio,
            dtype=np.float32
        ).squeeze()

    raise TypeError(
        f"Unsupported audio object: {type(entry)}"
    )


def compatible_audio_generator(
    self,
    split=None,
    repeat=1
):

    if split is None:
        clip_list = self.clips
    else:
        clip_list = self.split_clips[split]

    for _ in range(repeat):

        for clip in clip_list:

            clip_audio = audio_entry_to_numpy(
                clip["audio"]
            )

            if self.remove_silence:
                clip_audio = self.remove_silence_function(
                    clip_audio
                )

            if self.trim_zeros:
                clip_audio = np.trim_zeros(
                    clip_audio
                )

            if self.trimmed_clip_duration_s:

                total_samples = int(
                    self.trimmed_clip_duration_s *
                    16000
                )

                clip_audio = clip_audio[
                    :total_samples
                ]

            clip_audio = self.repeat_clip(
                clip_audio
            )

            yield clip_audio


def compatible_get_random_clip(self):

    rand_audio_entry = random.choice(
        self.clips
    )

    clip_audio = audio_entry_to_numpy(
        rand_audio_entry["audio"]
    )

    if self.remove_silence:
        clip_audio = self.remove_silence_function(
            clip_audio
        )

    if self.trim_zeros:
        clip_audio = np.trim_zeros(
            clip_audio
        )

    if self.trimmed_clip_duration_s:

        total_samples = int(
            self.trimmed_clip_duration_s *
            16000
        )

        clip_audio = clip_audio[
            :total_samples
        ]

    return self.repeat_clip(
        clip_audio
    )


Clips.audio_generator = compatible_audio_generator
Clips.get_random_clip = compatible_get_random_clip


print("✅ AudioDecoder compatibility enabled")


# ------------------------------------------------------------
# Verify our datasets still exist
# ------------------------------------------------------------

assert os.path.isdir(
    "/content/generated_samples"
), "Hey Elli samples missing"

assert os.path.isdir(
    "/content/mit_rirs"
), "MIT RIRs missing"

assert os.path.isdir(
    "/content/audioset_16k"
), "AudioSet backgrounds missing"


# ------------------------------------------------------------
# Rebuild Hey Elli clip dataset
# ------------------------------------------------------------

clips = Clips(
    input_directory="/content/generated_samples",
    file_pattern="*.wav",
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)


# ------------------------------------------------------------
# Rebuild augmentation pipeline
# ------------------------------------------------------------

augmenter = Augmentation(

    augmentation_duration_s=3.2,

    augmentation_probabilities={
        "SevenBandParametricEQ": 0.10,
        "TanhDistortion":       0.10,
        "PitchShift":           0.10,
        "BandStopFilter":       0.10,
        "AddColorNoise":        0.10,
        "AddBackgroundNoise":   0.75,
        "Gain":                 1.00,
        "RIR":                  0.50,
    },

    impulse_paths=[
        "/content/mit_rirs"
    ],

    background_paths=[
        "/content/audioset_16k"
    ],

    background_min_snr_db=-5,
    background_max_snr_db=10,

    min_jitter_s=0.195,
    max_jitter_s=0.205,
)


print("✅ Hey Elli augmentation rebuilt")


# ------------------------------------------------------------
# Fresh Step 5 output
# ------------------------------------------------------------

output_dir = (
    "/content/generated_augmented_features"
)

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(
    output_dir,
    exist_ok=True
)


# ------------------------------------------------------------
# Generate TRAIN / VALIDATION / TEST features
# ------------------------------------------------------------

split_settings = [

    # folder, Clips split, repeat count, sliding frames
    ("training",   "train",      2, 10),
    ("validation", "validation", 1, 10),
    ("testing",    "test",       1, 1),
]


for (
    folder_name,
    split_name,
    repetition,
    slide_frames
) in split_settings:

    print()
    print(
        "================================"
    )

    print(
        "Generating:",
        folder_name
    )

    print(
        "================================"
    )

    split_dir = os.path.join(
        output_dir,
        folder_name
    )

    os.makedirs(
        split_dir,
        exist_ok=True
    )

    spectrograms = SpectrogramGeneration(
        clips=clips,
        augmenter=augmenter,
        slide_frames=slide_frames,
        step_ms=10,
    )

    RaggedMmap.from_generator(

        out_dir=os.path.join(
            split_dir,
            "wakeword_mmap"
        ),

        sample_generator=(
            spectrograms.spectrogram_generator(
                split=split_name,
                repeat=repetition
            )
        ),

        batch_size=100,
        verbose=True,
    )

    print(
        "✅",
        folder_name,
        "complete"
    )


print()
print(
    "========================================"
)

print(
    "✅ HEY ELLI STEP 5 COMPLETE"
)

print(
    "========================================"
)

print(
    "Training features: READY"
)

print(
    "Validation features: READY"
)

print(
    "Testing features: READY"
)

✅ AudioDecoder compatibility enabled
✅ Hey Elli augmentation rebuilt

Generating: training


0it [00:00, ?it/s]

✅ training complete

Generating: validation


0it [00:00, ?it/s]

✅ validation complete

Generating: testing


0it [00:00, ?it/s]

✅ testing complete

✅ HEY ELLI STEP 5 COMPLETE
Training features: READY
Validation features: READY
Testing features: READY


In [9]:
import shutil

total, used, free = shutil.disk_usage("/content")

print(f"Total disk: {total/1024**3:.1f} GB")
print(f"Used disk : {used/1024**3:.1f} GB")
print(f"Free disk : {free/1024**3:.1f} GB")

Total disk: 112.6 GB
Used disk : 50.2 GB
Free disk : 62.4 GB


In [7]:
# ============================================================
# KIRA FIX - pymicro-features 2.x compatibility
# ============================================================

from pymicro_features import MicroFrontend
import importlib.metadata

print(
    "pymicro-features:",
    importlib.metadata.version("pymicro-features")
)

# microWakeWord expects the old v1 method name.
# pymicro-features v2 renamed it to process_samples().
if (
    not hasattr(MicroFrontend, "ProcessSamples")
    and hasattr(MicroFrontend, "process_samples")
):
    MicroFrontend.ProcessSamples = MicroFrontend.process_samples

print(
    "ProcessSamples available:",
    hasattr(MicroFrontend, "ProcessSamples")
)

# Tiny sanity test
import numpy as np

frontend = MicroFrontend()

test_audio = np.zeros(
    160,
    dtype=np.int16
).tobytes()

result = frontend.ProcessSamples(
    test_audio
)

print("✅ MicroFrontend compatibility fixed")
print("Samples read:", result.samples_read)
print("Feature count:", len(result.features))

pymicro-features: 2.0.2
ProcessSamples available: True
✅ MicroFrontend compatibility fixed
Samples read: 160
Feature count: 0


## 📥 Step 6: Download Negative Datasets

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
    <p><b>What this step does:</b> Downloads pre-generated spectrogram features for various negative datasets.</p>
    <p><b>Why this matters:</b> Negative samples help the model learn what is NOT your wake word, reducing false activations.</p>
    <p><b>Datasets included:</b></p>
    <ul>
        <li><code>dinner_party</code> - Conversations in a dinner party setting</li>
        <li><code>dinner_party_eval</code> - Separate evaluation set of dinner party audio</li>
        <li><code>no_speech</code> - Environmental sounds without speech</li>
        <li><code>speech</code> - Various speech samples</li>
    </ul>
</div>

In [10]:
# Downloads pre-generated spectrogram features (made for microWakeWord in
# particular) for various negative datasets. This can be slow!

output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        link = link_root + fname

        zip_path = f"negative_datasets/{fname}"
        !wget -O {zip_path} {link}
        !unzip -q {zip_path} -d {output_dir}

--2026-09-14 11:16:45--  https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/dinner_party.zip
Resolving huggingface.co (huggingface.co)... 99.86.101.36, 99.86.101.39, 99.86.101.64, ...
Connecting to huggingface.co (huggingface.co)|99.86.101.36|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65e327bc1445a768ed343b8c/228d7e72cd5fdc4e6e57da36b88a4c227d34cb8dc44041078b4c4b65dc75848d?X-Xet-Cas-Uid=public&user_id=public&response-content-type=application%2Fzip&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27dinner_party.zip%3B+filename%3D%22dinner_party.zip%22%3B&Expires=1789388206&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjVlMzI3YmMxNDQ1YTc2OGVkMzQzYjhjLzIyOGQ3ZTcyY2Q1ZmRjNGU2ZTU3ZGEzNmI4OGE0YzIyN2QzNGNiOGRjNDQwNDEwNzhiNGM0YjY1ZGM3NTg0OGRcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ1c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LXR5cGU9KiZyZXNwb25zZS1jb250ZW5

In [11]:
import os

paths = [
    "/content/generated_augmented_features/training/wakeword_mmap",
    "/content/generated_augmented_features/validation/wakeword_mmap",
    "/content/generated_augmented_features/testing/wakeword_mmap",
    "/content/negative_datasets/speech",
    "/content/negative_datasets/no_speech",
    "/content/negative_datasets/dinner_party",
    "/content/negative_datasets/dinner_party_eval",
]

for p in paths:
    print("✅" if os.path.exists(p) else "❌", p)

✅ /content/generated_augmented_features/training/wakeword_mmap
✅ /content/generated_augmented_features/validation/wakeword_mmap
✅ /content/generated_augmented_features/testing/wakeword_mmap
✅ /content/negative_datasets/speech
✅ /content/negative_datasets/no_speech
✅ /content/negative_datasets/dinner_party
✅ /content/negative_datasets/dinner_party_eval


## ⚙️ Step 7: Configure Training Parameters

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
    <p><b>What this step does:</b> Creates a YAML configuration file that controls the training process.</p>
    <p><b>Why this matters:</b> These hyperparameters can make a huge difference in model quality.</p>
    <p><b>Key parameters to experiment with:</b></p>
    <ul>
        <li><code>sampling_weight</code> - Controls how often samples from each dataset are used in training</li>
        <li><code>penalty_weight</code> - Controls how much incorrect predictions from each dataset are penalized</li>
        <li><code>training_steps</code> - Number of training iterations (increase for potentially better models)</li>
        <li><code>positive_class_weight</code> and <code>negative_class_weight</code> - Balance between false positives and false negatives</li>
    </ul>
</div>

In [15]:
# Save a yaml config that controls the training process
# These hyperparamters can make a huge different in model quality.
# Experiment with sampling and penalty weights and increasing the number of
# training steps.

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = (
    "trained_models/wakeword"
)


# Each feature_dir should have at least one of the following folders with this structure:
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight: Weight for choosing a spectrogram from this set in the batch
#  penalty_weight: Penalizing weight for incorrect predictions from this set
#  truth: Boolean whether this set has positive samples or negative samples
#  truncation_strategy = If spectrograms in the set are longer than necessary for training, how are they truncated
#       - random: choose a random portion of the entire spectrogram - useful for long negative samples
#       - truncate_start: remove the start of the spectrogram
#       - truncate_end: remove the end of the spectrogram
#       - split: Split the longer spectrogram into separate spectrograms offset by 100 ms. Only for ambient sets

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    { # Only used for validation and testing
        "features_dir": "negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

# Number of training steps in each iteration - various other settings are configured as lists that corresponds to different steps
config["training_steps"] = [10000]

# Penalizing weight for incorrect class predictions - lists that correspond to training steps
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

config["learning_rates"] = [
    0.001,
]  # Learning rates for Adam optimizer - list that corresponds to training steps
config["batch_size"] = 128

config["time_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["time_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["freq_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps

config["eval_step_interval"] = (
    500  # Test the validation sets after every this many steps
)
config["clip_duration_ms"] = (
    1500  # Maximum length of wake word that the streaming model will accept
)

# The best model weights are chosen first by minimizing the specified minimization metric below the specified target_minimization
# Once the target has been met, it chooses the maximum of the maximization metric. Set 'minimization_metric' to None to only maximize
# Available metrics:
#   - "loss" - cross entropy error on validation set
#   - "accuracy" - accuracy of validation set
#   - "recall" - recall of validation set
#   - "precision" - precision of validation set
#   - "false_positive_rate" - false positive rate of validation set
#   - "false_negative_rate" - false negative rate of validation set
#   - "ambient_false_positives" - count of false positives from the split validation_ambient set
#   - "ambient_false_positives_per_hour" - estimated number of false positives per hour on the split validation_ambient set
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # Set to None to disable

config["maximization_metric"] = "average_viable_recall"

with open(os.path.join("training_parameters.yaml"), "w") as file:
    documents = yaml.dump(config, file)

In [16]:
import os

print(
    "✅ training_parameters.yaml ready"
    if os.path.exists("/content/training_parameters.yaml")
    else "❌ training_parameters.yaml missing"
)

✅ training_parameters.yaml ready


In [17]:
# KIRA - HEY ELLI - STEP 8 PREFLIGHT

import os
import tensorflow as tf

# Make subprocesses use our full source checkout
%env PYTHONPATH=/content/microWakeWord

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

required = [
    "/content/training_parameters.yaml",
    "/content/generated_augmented_features",
    "/content/negative_datasets/speech",
    "/content/negative_datasets/no_speech",
    "/content/negative_datasets/dinner_party",
    "/content/negative_datasets/dinner_party_eval",
]

for p in required:
    print("✅" if os.path.exists(p) else "❌", p)

print("\nTesting training module...")
!python -c "import microwakeword.model_train_eval as m; print('✅ Module:', m.__file__)"

env: PYTHONPATH=/content/microWakeWord
TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ /content/training_parameters.yaml
✅ /content/generated_augmented_features
✅ /content/negative_datasets/speech
✅ /content/negative_datasets/no_speech
✅ /content/negative_datasets/dinner_party
✅ /content/negative_datasets/dinner_party_eval

Testing training module...
    pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
✅ Module: /content/microWakeWord/microwakeword/model_train_eval.py


## 🚀 Step 8: Train the Model

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
    <p><b>What this step does:</b> Trains the neural network model using the data and configuration from previous steps.</p>
    <p><b>Expected time:</b> 30+ minutes (much faster with a GPU)</p>
    <p><b>What to expect:</b> The training process will print progress updates. When finished, it will convert the model to a streaming version suitable for on-device detection.</p>
    <p><b>Key parameters to modify:</b></p>
    <ul>
        <li><code>--train 1</code> - Set to 0 to only convert and test the best-weighted model without training</li>
        <li>Neural network architecture parameters at the end of the command</li>
    </ul>
</div>

In [25]:
!PYTHONPATH=/content/microWakeWord python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

    pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
INFO:absl:Loading and analyzing data sets.
2026-09-14 12:20:13.148963: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1789388413.150433   36374 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (128, 204, 4

In [29]:
!PYTHONPATH=/content/microWakeWord python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

    pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
INFO:absl:Loading and analyzing data sets.
2026-09-14 12:28:57.343532: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1789388937.345014   38627 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (128, 204, 4

In [30]:
# ============================================================
# KIRA - BACK UP TRAINED "HEY ELLI" MODEL
# ============================================================

import os
import glob
import shutil
from pathlib import Path

print("Searching for trained TFLite models...\n")

models = glob.glob(
    "/content/**/*.tflite",
    recursive=True
)

for model in models:
    size_kb = os.path.getsize(model) / 1024
    print(f"{size_kb:8.1f} KB  {model}")

print("\nTotal TFLite models:", len(models))

Searching for trained TFLite models...

    59.5 KB  /content/microWakeWord/examples/hey_nexus.tflite
    59.5 KB  /content/microWakeWord/examples/hey_nexus_v2.tflite
    59.5 KB  /content/trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite

Total TFLite models: 3


In [31]:
# ============================================================
# KIRA - PACKAGE TRAINED "HEY ELLI" MODEL
# ============================================================

from pathlib import Path
import shutil
import hashlib

package = Path("/content/KIRA_Hey_Elli")
if package.exists():
    shutil.rmtree(package)

package.mkdir()

# ------------------------------------------------------------
# Core trained model
# ------------------------------------------------------------

model = Path(
    "/content/trained_models/wakeword/"
    "tflite_stream_state_internal_quant/"
    "stream_state_internal_quant.tflite"
)

assert model.exists(), "❌ Trained Hey Elli model missing"

shutil.copy(
    model,
    package / "hey_elli.tflite"
)


# ------------------------------------------------------------
# ROC / threshold results
# ------------------------------------------------------------

roc = Path(
    "/content/trained_models/wakeword/"
    "tflite_stream_state_internal_quant/"
    "tflite_streaming_roc.txt"
)

if roc.exists():
    shutil.copy(
        roc,
        package / "hey_elli_roc.txt"
    )


# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

training_params = Path(
    "/content/training_parameters.yaml"
)

if training_params.exists():
    shutil.copy(
        training_params,
        package / "training_parameters.yaml"
    )


training_config = Path(
    "/content/trained_models/wakeword/"
    "training_config.yaml"
)

if training_config.exists():
    shutil.copy(
        training_config,
        package / "training_config.yaml"
    )


# ------------------------------------------------------------
# Preserve weights too, if available
# ------------------------------------------------------------

for filename in [
    "best_weights.weights.h5",
    "last_weights.weights.h5"
]:
    p = Path(
        "/content/trained_models/wakeword"
    ) / filename

    if p.exists():
        shutil.copy(
            p,
            package / filename
        )


# ------------------------------------------------------------
# Model checksum + info
# ------------------------------------------------------------

data = model.read_bytes()

sha256 = hashlib.sha256(data).hexdigest()

info = f"""KIRA Custom Wake Word Model

Wake word: Hey Elli
Model type: microWakeWord
Inference: Streaming quantized TFLite
Model size: {len(data) / 1024:.1f} KB

Recommended first test thresholds:
0.86 = safer / fewer false activations
0.84 = more sensitive

SHA256:
{sha256}
"""

(package / "MODEL_INFO.txt").write_text(info)


# ------------------------------------------------------------
# ZIP EVERYTHING
# ------------------------------------------------------------

zip_path = shutil.make_archive(
    "/content/KIRA_Hey_Elli_BACKUP",
    "zip",
    "/content/KIRA_Hey_Elli"
)

print("===================================")
print("✅ HEY ELLI MODEL BACKUP COMPLETE")
print("===================================")

for f in package.iterdir():
    print(
        f"{f.name:32s}",
        f"{f.stat().st_size / 1024:.1f} KB"
    )

print()
print("ZIP:", zip_path)

✅ HEY ELLI MODEL BACKUP COMPLETE
last_weights.weights.h5          430.8 KB
hey_elli.tflite                  59.5 KB
best_weights.weights.h5          430.8 KB
hey_elli_roc.txt                 0.4 KB
training_config.yaml             1.9 KB
MODEL_INFO.txt                   0.3 KB
training_parameters.yaml         1.1 KB

ZIP: /content/KIRA_Hey_Elli_BACKUP.zip


In [32]:
from google.colab import files
files.download("/content/KIRA_Hey_Elli_BACKUP.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
# ============================================================
# KIRA - ARCHIVE CURRENT HEY ELLI TRAINING WORKSPACE
# ============================================================

import os
import shutil

moves = {
    "/content/generated_samples":
        "/content/hey_elli_generated_samples",

    "/content/generated_augmented_features":
        "/content/hey_elli_augmented_features",

    "/content/trained_models":
        "/content/hey_elli_trained_models",
}

for src, dst in moves.items():

    if os.path.exists(src):

        if os.path.exists(dst):
            shutil.rmtree(dst)

        shutil.move(src, dst)

        print("✅", src, "→", dst)

    else:
        print("ℹ️ Not present:", src)

print()
print("✅ Hey Elli workspace protected")

✅ /content/generated_samples → /content/hey_elli_generated_samples
✅ /content/generated_augmented_features → /content/hey_elli_augmented_features
✅ /content/trained_models → /content/hey_elli_trained_models

✅ Hey Elli workspace protected


In [27]:
# ============================================================
# KIRA / HEY ELLI
# STEP-500 VALIDATION MEMORY FIX V2
# Caps BOTH validation and validation_ambient
# Training data remains completely unchanged.
# ============================================================

from pathlib import Path

path = Path("/content/microWakeWord/microwakeword/data.py")
text = path.read_text()

# Remove our previous V1 guard if present
old_v1 = '''
                    # KIRA / Colab memory guard:
                    # Keep training untouched, but cap ordinary validation
                    # so Step-500 evaluation does not create a ~380 MB array.
                    if mode == "validation" and len(data) >= 8000:
                        break

                if mode == "validation" and len(data) >= 8000:
                    break
'''

if old_v1 in text:
    text = text.replace(old_v1, "\n")

# Find the normal append block
old = '''                for spectrogram in generator:
                    data.append(spectrogram)
                    labels.append(provider.label)
                    weights.append(provider.penalty_weight)
'''

new = '''                for spectrogram in generator:
                    data.append(spectrogram)
                    labels.append(provider.label)
                    weights.append(provider.penalty_weight)

                    # KIRA / Colab validation memory guard V2
                    # Training/testing are NOT affected.
                    if mode == "validation" and len(data) >= 4000:
                        break

                    if mode == "validation_ambient" and len(data) >= 2000:
                        break

                if mode == "validation" and len(data) >= 4000:
                    break

                if mode == "validation_ambient" and len(data) >= 2000:
                    break
'''

if "KIRA / Colab validation memory guard V2" in text:
    print("✅ V2 memory fix already installed")

elif old in text:
    text = text.replace(old, new)
    path.write_text(text)
    print("✅ V2 memory fix installed")

else:
    raise RuntimeError(
        "Could not find the expected data.py block. "
        "Do NOT start training yet."
    )

# Verify syntax
compile(path.read_text(), str(path), "exec")

print("✅ data.py syntax OK")
print("✅ Training data: FULL")
print("✅ Validation cap: 4000 samples")
print("✅ Ambient validation cap: 2000 samples")

✅ V2 memory fix already installed
✅ data.py syntax OK
✅ Training data: FULL
✅ Validation cap: 4000 samples
✅ Ambient validation cap: 2000 samples


In [28]:
from pathlib import Path

text = Path(
    "/content/microWakeWord/microwakeword/data.py"
).read_text()

print(
    "✅ V2 patch confirmed"
    if "KIRA / Colab validation memory guard V2" in text
    else "❌ PATCH MISSING"
)

✅ V2 patch confirmed


In [24]:
# ============================================================
# KIRA - COLAB VALIDATION MEMORY FIX
# Limit validation samples only; training data stays FULL.
# ============================================================

from pathlib import Path

path = Path("/content/microWakeWord/microwakeword/data.py")
text = path.read_text()

old = """                for spectrogram in generator:
                    data.append(spectrogram)
                    labels.append(provider.label)
                    weights.append(provider.penalty_weight)
"""

new = """                for spectrogram in generator:
                    data.append(spectrogram)
                    labels.append(provider.label)
                    weights.append(provider.penalty_weight)

                    # KIRA / Colab memory guard:
                    # Keep training untouched, but cap ordinary validation
                    # so Step-500 evaluation does not create a ~380 MB array.
                    if mode == "validation" and len(data) >= 8000:
                        break

                if mode == "validation" and len(data) >= 8000:
                    break
"""

if old in text:
    text = text.replace(old, new)
    path.write_text(text)
    print("✅ Validation memory guard installed")
elif "KIRA / Colab memory guard" in text:
    print("✅ Validation memory guard already installed")
else:
    print("❌ Expected code block not found")

# syntax check
compile(path.read_text(), str(path), "exec")
print("✅ data.py syntax OK")

✅ Validation memory guard installed
✅ data.py syntax OK


In [19]:
# ============================================================
# KIRA / HEY ELLI
# Fix microWakeWord for current Keras/NumPy metric outputs
# ============================================================

from pathlib import Path

path = Path("/content/microWakeWord/microwakeword/train.py")

text = path.read_text()

replacements = {
    'result["fp"].numpy()':
        'np.asarray(result["fp"])',

    'ambient_predictions["tp"].numpy()':
        'np.asarray(ambient_predictions["tp"])',

    'ambient_predictions["fp"].numpy()':
        'np.asarray(ambient_predictions["fp"])',

    'ambient_predictions["fn"].numpy()':
        'np.asarray(ambient_predictions["fn"])',
}

changed = 0

for old, new in replacements.items():
    if old in text:
        text = text.replace(old, new)
        changed += 1
        print("✅ Patched:", old)
    elif new in text:
        print("✅ Already patched:", new)
    else:
        print("⚠️ Not found:", old)

path.write_text(text)

# Verify Python syntax
compile(
    path.read_text(),
    str(path),
    "exec"
)

print()
print("✅ train.py compatibility patch complete")
print("Replacements made:", changed)

✅ Patched: result["fp"].numpy()
✅ Patched: ambient_predictions["tp"].numpy()
✅ Patched: ambient_predictions["fp"].numpy()
✅ Patched: ambient_predictions["fn"].numpy()

✅ train.py compatibility patch complete
Replacements made: 4


## 📤 Step 9: Export the Model

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
    <p><b>What this step does:</b> Downloads the trained TFLite model file for use with ESPHome.</p>
    <p><b>Next steps:</b></p>
    <ol>
        <li>Create a model manifest JSON file based on the training results</li>
        <li>Adjust the probability threshold based on test results</li>
        <li>Upload both files to your ESPHome device</li>
    </ol>
    <p><b>Resources:</b></p>
    <ul>
        <li><a href="https://esphome.io/components/micro_wake_word">ESPHome documentation</a></li>
        <li><a href="https://github.com/esphome/micro-wake-word-models/tree/main/models/v2">Example model configurations</a></li>
    </ul>
</div>

In [ ]:
# Downloads the tflite model file. To use on the device, you need to write a
# Model JSON file. See https://esphome.io/components/micro_wake_word for the
# documentation and
# https://github.com/esphome/micro-wake-word-models/tree/main/models/v2 for
# examples. Adjust the probability threshold based on the test results obtained
# after training is finished. You may also need to increase the Tensor arena
# model size if the model fails to load.

import os

# Get the model file path
model_path = "trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"

# Check if running in a Jupyter environment
try:
    from google.colab import files
    # If in Colab, use files.download
    files.download(model_path)
    print(f"Model downloaded from {model_path}")
except ImportError:
    # If not in Colab, just print the path
    print(f"\nModel saved at: {os.path.abspath(model_path)}")
    print("\nTo use this model with ESPHome:")
    print("1. Create a model manifest JSON file")
    print("2. Copy both files to your ESPHome configuration directory")
    print("3. Configure ESPHome to use the model")

## 🎉 Congratulations!

<div style="background-color: #dff0d8; padding: 15px; border-radius: 10px; border-left: 5px solid #3c763d; margin-bottom: 20px;">
    <h3 style="margin-top: 0; color: #3c763d;">You've Successfully Trained a Wake Word Model!</h3>
    <p>You've completed all the steps to train a custom wake word model with microWakeWord. Here's what you can do next:</p>
    <ol>
        <li><b>Test your model</b> - Try different probability thresholds to balance between detection rate and false positives</li>
        <li><b>Experiment</b> - Try different training parameters to improve your model</li>
        <li><b>Deploy to ESPHome</b> - Use your model on an ESP32 device</li>
    </ol>
    <p>Remember that wake word model training is an iterative process. You may need to adjust parameters and retrain several times to get the best results for your specific use case.</p>
</div>

### Example ESPHome Configuration

```yaml
# Wake word configuration
micro_wake_word:
  model_file: "stream_state_internal_quant.tflite"
  model_name: "my_wake_word"
  probability_cutoff: 0.5  # Adjust based on training results
  
binary_sensor:
  - platform: micro_wake_word
    name: "Wake Word Detected"
    id: wake_word
    model_id: my_wake_word
    
# Optional - add a text-to-speech response
esphome:
  on_boot:
    priority: -100
    then:
      - delay: 5s
      - logger.log: "Wake word detection ready"
```